**What:**
- Load the merged table merged_85ms.csv and the 13 saved events b4_if_events_gap5.csv from the Isolation Forest notebook
- Check that the timestamps are valid and ordered, that there is at least one event and that every event lies inside the recording
- Print the data shape, the recording period and the event table

**Why:**
- All hypothesis tests below compare activations with these event windows, so the events must match the recording before we use them
- The checks stop the cell with an error if they do not hold

**Output:**
- Data: (5945, 202), recording 2019-09-26 18:21:14.940 to 18:29:40.180 UTC
- 13 events loaded, from event 1 (18:21:16.045 to 18:21:20.040, 47 flagged rows) to event 13 (18:28:49.690 to 18:28:53.600, 47 flagged rows)
- both_share is 1.000 for 10 events. It is 0.875 (event 4), 0.923 (event 5), 0.950 (event 7) and 0.000 (event 11)

**Next:** H1, find ON periods of output signals that are longer than usual.

In [15]:
# Cell 1: load the data and the saved events, check them.
import re  # regular expressions, used in the next cell
import numpy as np  # numerical tools
import pandas as pd  # tables

FOLDER = '/Users/bala/Desktop/Albstadt Uni/Project B4/progressed work/'
data = pd.read_csv(FOLDER + 'merged_85ms.csv')  # merged 85 ms table
events = pd.read_csv(FOLDER + 'b4_if_events_gap5.csv')  # 13 IF events
events['start_utc'] = pd.to_datetime(  # event start as a timestamp
    events['start_utc'], utc=True, format='ISO8601')
events['end_utc'] = pd.to_datetime(  # event end as a timestamp
    events['end_utc'], utc=True, format='ISO8601')
utc = pd.to_datetime(  # recorded times of the data rows
    data['utc_time'], utc=True, format='ISO8601').reset_index(drop=True)

assert utc.is_monotonic_increasing and not utc.isna().any(), \
    'utc_time must be valid and ordered.'  # valid and ordered times
assert len(events) > 0, 'No events in the saved file.'  # events exist
assert ((events['start_utc'] >= utc.iloc[0]).all()  # inside the recording
        and (events['end_utc'] <= utc.iloc[-1]).all()), \
    'Some events lie outside the recording.'

print(f'Data: {data.shape} | recording {utc.iloc[0]} to {utc.iloc[-1]}')
print(f'Events loaded: {len(events)}')
print(events.assign(  # readable times for printing
    start_utc=events['start_utc'].dt.strftime('%H:%M:%S.%f').str[:-3],
    end_utc=events['end_utc'].dt.strftime('%H:%M:%S.%f').str[:-3])
      .to_string(index=False))  # event table


Data: (5945, 202) | recording 2019-09-26 18:21:14.940000+00:00 to 2019-09-26 18:29:40.180000+00:00
Events loaded: 13
 event    start_utc      end_utc  flagged_rows  intervals  lowest_score  both_share  seconds
     1 18:21:16.045 18:21:20.040            47          2       -0.5059       1.000    3.995
     2 18:21:46.305 18:21:47.495            15          1       -0.5320       1.000    1.190
     3 18:22:57.450 18:22:58.895            13          2       -0.5064       1.000    1.445
     4 18:23:18.190 18:23:24.055            35          4       -0.5530       0.875    5.865
     5 18:24:16.245 18:24:18.030            21          2       -0.5177       0.923    1.785
     6 18:25:22.715 18:25:25.180            30          1       -0.5416       1.000    2.465
     7 18:25:31.980 18:25:38.695            57          2       -0.5212       0.950    6.715
     8 18:26:17.965 18:26:18.050             2          1       -0.5033       1.000    0.085
     9 18:26:58.170 18:26:59.530            16

**What:**
- H1: find the output signals that stayed ON longer than usual. We take the 31 output bits: Siemens outputs `A<number>.<bit>` (without the communication and non-step outputs `A200`, `A201`, `A6.`, `A0.1`, `A1.5`, `A1.6`, `A4.1`) and Beckhoff outputs `<name>.Q<number>_<name>` (without the long copies `_horizontal_`, `_vertical_`, `_cantilever_`)
- Measure every complete ON-to-OFF episode in seconds
- The usual duration of a signal is its own median, used only if the signal has at least 5 episodes and a median of at least 0.25 s
- Flag an episode if it is at least 25% above the median and at least 2 s longer. Whole episodes are measured, not cut at the event window
- For each event count the episodes that overlap it, how many were compared and how many are flagged

**Why:**
- A signal that stays ON much longer than its own usual time can point to a slow step. We call the flags candidates, not confirmed losses
- The checks stop the cell if an output name is not recognised or a signal is not 0/1

**Output:**
- 31 output bits, 497 complete activations, 487 compared, 13 flagged in the whole recording
- Flagged activations that overlap an event: event 7 `WH.Q3_horRack` (13.600 s, median 3.103 s, 10.497 s extra) and `A1.7` (18.105 s, median 9.265 s, 8.840 s extra); event 10 `A0.7` (7.990 s, 2.210 s extra); event 11 `WH.Q7_cantForw` (13.430 s, 10.370 s extra); event 13 `WH.Q3_horRack` (10.285 s, 7.183 s extra)
- Event 1 overlaps no activation. Events 4 and 10 overlap 13 each
- Saved: `b4_h1_per_event_summary.csv` and `b4_h1_per_event_flagged.csv`

**Next:** H2, robot transfer timing.

In [16]:
# Cell 2: for each event, did an output signal stay ON longer than usual?
# Complete ON/OFF episodes only; baseline = the signal's own median
# (>= 5 repetitions, median >= 0.25 s); flag = at least 25% over the median
# AND at least 2 s extra. Full episodes are measured, not clipped.
not_steps = ('A200', 'A201', 'A6.', 'A0.1', 'A1.5', 'A1.6', 'A4.1')  # skip
long_copies = ('_horizontal_', '_vertical_', '_cantilever_')  # skip
siemens = [c for c in data  # Siemens outputs A<number>.<bit>
           if c.startswith('A') and not c.startswith(not_steps)
           and '.' in c and c.split('.')[0][1:].isdigit()
           and c.split('.')[1].isdigit() and int(c.split('.')[1]) <= 7]
beckhoff = [c for c in data  # Beckhoff outputs <name>.Q<number>_<name>
            if '.Q' in c and not any(s in c for s in long_copies)]

eps = []  # one entry per complete ON/OFF episode
for c in siemens + beckhoff:  # every output bit
    s = data[c].reset_index(drop=True)  # its values
    assert s.isin([0, 1]).all(), f'{c}: non-binary values.'  # 0/1 only
    on = s.eq(1).astype(int)  # 1 while the signal is ON
    d = on.diff()  # +1 = switched ON, -1 = switched OFF
    ends = np.flatnonzero(d.eq(-1))  # row numbers of the OFF edges
    for st in np.flatnonzero(d.eq(1)):  # each ON edge
        after = ends[ends > st]  # OFF edges after it
        if len(after):  # keep only complete episodes
            eps.append((c, utc.iloc[st], utc.iloc[after[0]],
                        (utc.iloc[after[0]] - utc.iloc[st]).total_seconds()))
h1 = pd.DataFrame(  # all episodes
    eps, columns=['signal', 'on_utc', 'off_utc', 'activation_s'])
base = h1.groupby('signal')['activation_s'].agg(  # per-signal baseline
    repetitions='size', median_s='median')
h1 = h1.join(base, on='signal')  # add the baseline to each episode
h1['usable'] = (h1['repetitions'] >= 5) & (h1['median_s'] >= 0.25)  # compare?
h1['extra_s'] = h1['activation_s'] - h1['median_s']  # time above the median
h1['H1_flag'] = (h1['usable']  # long: 25% over the median and 2 s extra
                 & (h1['activation_s'] >= 1.25 * h1['median_s'])
                 & (h1['extra_s'] >= 2))
print(f"{len(siemens) + len(beckhoff)} output bits | "
      f"complete activations: {len(h1)} | "
      f"compared: {int(h1['usable'].sum())} | "
      f"flagged in the whole recording: {int(h1['H1_flag'].sum())}")

rows, flagged = [], []  # per-event summary and flagged episodes
for _, e in events.iterrows():  # each event
    hit = h1[(h1['on_utc'] < e['end_utc'])  # episodes overlapping it
             & (h1['off_utc'] > e['start_utc'])]
    fl = hit[hit['H1_flag']]  # the flagged ones among them
    rows.append([int(e['event']),
                 e['start_utc'].strftime('%H:%M:%S.%f')[:-3],
                 e['end_utc'].strftime('%H:%M:%S.%f')[:-3],
                 int(e['flagged_rows']), e['both_share'], len(hit),
                 int(hit['usable'].sum()), len(fl)])
    if len(fl):  # remember flagged episodes with their event
        flagged.append(fl.assign(event=int(e['event'])))
summary = pd.DataFrame(rows, columns=[  # H1 result per event
    'event', 'start_utc', 'end_utc', 'flagged_rows', 'both_share',
    'activations_overlapping', 'compared', 'H1_flagged'])
print('\nH1 per event:')
print(summary.to_string(index=False))  # per-event table
if flagged:  # any flagged episode overlaps an event
    fl_all = pd.concat(flagged)[[
        'event', 'signal', 'on_utc', 'off_utc',
        'activation_s', 'median_s', 'extra_s']]
    print('\nH1-flagged activations overlapping an event (full episodes):')
    print(fl_all.sort_values(['event', 'on_utc']).to_string(
        index=False, float_format=lambda x: f'{x:.3f}'))
    fl_all.to_csv(FOLDER + 'b4_h1_per_event_flagged.csv', index=False)
summary.to_csv(FOLDER + 'b4_h1_per_event_summary.csv', index=False)  # save
print('\nSaved: b4_h1_per_event_summary.csv '
      '(+ b4_h1_per_event_flagged.csv if any flag).')
print('H1 flag = ON longer than that signal usually is. '
      'Candidates only, not confirmed losses.')


31 output bits | complete activations: 497 | compared: 487 | flagged in the whole recording: 13

H1 per event:
 event    start_utc      end_utc  flagged_rows  both_share  activations_overlapping  compared  H1_flagged
     1 18:21:16.045 18:21:20.040            47       1.000                        0         0           0
     2 18:21:46.305 18:21:47.495            15       1.000                        2         2           0
     3 18:22:57.450 18:22:58.895            13       1.000                        2         2           0
     4 18:23:18.190 18:23:24.055            35       0.875                       13        13           0
     5 18:24:16.245 18:24:18.030            21       0.923                        3         3           0
     6 18:25:22.715 18:25:25.180            30       1.000                        6         6           0
     7 18:25:31.980 18:25:38.695            57       0.950                        2         2           2
     8 18:26:17.965 18:26:18.050         

**What:**
- H2: robot transfer timing, in two parts, using the signals as timing proxies
- Part A: for every complete belt-start interval (`WH.Q1_beltForward`), the time from the belt start to the first grip indication (`VG.Q7_compressorON`)
- Part B: for every oven start (`A1.1`), the gap to the latest grip release (end of `VG.Q7_compressorON`), with a tolerance of 0.1 s
- Flag a time that is at least 1 s above the median of its own part (the screening rule we use, not a validated fault cutoff)
- For each event, count the Part A and Part B segments that overlap it and how many of them are flagged
- The helper `complete()` returns the complete ON/OFF episodes of a signal. Cell 4 uses it too

**Why:**
- A long wait between belt start and grip, or between release and oven start, can show that the robot transfer waited. A flag means longer than usual, not a confirmed loss
- The checks stop the cell if a signal is not 0/1 or fewer than 2 belt activations exist

**Output:**
- Part A: 8 complete belt-start intervals, median 1.445 s, 1 flagged (interval 7: belt start 18:27:05.395, grip on 18:27:12.790, 7.395 s), none without a grip
- Part B: 9 release-to-oven pairs, median gap 1.700 s, 3 flagged: 16.405 s (release 18:22:38.580), 14.365 s (18:25:06.395) and 26.860 s (18:26:28.845)
- Per event: only event 4 overlaps one segment (a Part B gap, not flagged). No event overlaps a flagged segment
- Saved `b4_h2_per_event_flagged.csv` only if a flagged segment overlaps an event. Here that is not the case, so the file is not written

**Next:** H3, long pauses between pusher activations.

In [17]:
# Cell 3: H2 per event (robot transfer timing). Needs data, events, utc.
# Part A: belt start -> first grip indication, for every complete interval.
# Part B: grip release -> oven start (A1.1) gap. Flag rule for both:
# at least 1 s above the median (the inherited screening rule, not a
# validated fault cutoff). Signals are timing proxies.
def complete(column):  # complete ON/OFF episodes of one signal
    s = data[column].reset_index(drop=True)  # its values
    assert s.isin([0, 1]).all(), f'{column}: expected binary values.'
    d = s.eq(1).astype(int).diff()  # +1 = switched ON, -1 = switched OFF
    ends = np.flatnonzero(d.eq(-1))  # row numbers of the OFF edges
    out = []  # list of (on time, off time)
    for st in np.flatnonzero(d.eq(1)):  # each ON edge
        later = ends[ends > st]  # OFF edges after it
        if len(later):  # keep only complete episodes
            out.append((utc.iloc[st], utc.iloc[later[0]]))
    return out


belt = complete('WH.Q1_beltForward')  # belt forward episodes
grip = complete('VG.Q7_compressorON')  # grip indication episodes
oven = complete('A1.1')  # oven start episodes
assert len(belt) >= 2, 'Need two complete belt activations.'

# Part A
a_rows = []  # one row per belt-start interval
for k, ((b0, _), (b1, _)) in enumerate(zip(belt, belt[1:]), 1):
    g = [(on, off) for on, off in grip if b0 <= on < b1]  # grips inside it
    on, off = g[0] if g else (pd.NaT, pd.NaT)  # the first grip, if any
    a_rows.append({
        'interval': k, 'belt_start': b0, 'next_belt_start': b1,
        'grip_on': on, 'grip_off': off,
        'start_to_grip_s': (on - b0).total_seconds() if g else np.nan})
A = pd.DataFrame(a_rows)  # Part A table
med_a = A['start_to_grip_s'].median()  # usual belt-start-to-grip time
A['extra_s'] = A['start_to_grip_s'] - med_a  # time above the median
A['H2a_flag'] = A['extra_s'] >= 1.0  # flag: at least 1 s above
print(f'Part A: {len(A)} complete belt-start intervals | '
      f'median belt-start-to-grip {med_a:.3f} s | '
      f'flagged: {int(A["H2a_flag"].sum())} | '
      f'without grip: {int(A["grip_on"].isna().sum())}')

# Part B (release = grip OFF; oven start must be at or after it,
# 0.1 s tolerance)
TOL = pd.Timedelta(seconds=0.1)  # tolerance
b_rows = []  # one row per oven start
for o_on, _ in oven:  # each oven start
    rel = [off for _, off in grip if off <= o_on + TOL]  # earlier releases
    if rel:  # needs a release before it
        r = max(rel)  # the latest release
        b_rows.append({'release': r, 'oven_start': o_on,
                       'gap_s': (o_on - r).total_seconds()})
B = pd.DataFrame(b_rows)  # Part B table
med_b = B['gap_s'].median()  # usual release-to-oven gap
B['extra_s'] = B['gap_s'] - med_b  # time above the median
B['H2b_flag'] = B['extra_s'] >= 1.0  # flag: at least 1 s above
print(f'Part B: {len(B)} release-to-oven pairs | '
      f'median gap {med_b:.3f} s | flagged: {int(B["H2b_flag"].sum())}')

# Per event: any part A wait segment (belt start -> grip on) or part B gap
# (release -> oven start) overlapping it.
rows, details = [], []  # per-event summary and flagged segments
for _, e in events.iterrows():  # each event
    a = A[A['grip_on'].notna()  # Part A segments overlapping the event
          & (A['belt_start'] < e['end_utc'])
          & (A['grip_on'] > e['start_utc'])]
    b = B[(B['release'] < e['end_utc'])  # Part B gaps overlapping it
          & (B['oven_start'] > e['start_utc'])]
    rows.append([int(e['event']),
                 e['start_utc'].strftime('%H:%M:%S.%f')[:-3],
                 e['end_utc'].strftime('%H:%M:%S.%f')[:-3],
                 len(a), int(a['H2a_flag'].sum()),
                 len(b), int(b['H2b_flag'].sum())])
    for _, r in a[a['H2a_flag']].iterrows():  # flagged Part A segments
        details.append([int(e['event']), 'A belt->grip', r['belt_start'],
                        r['grip_on'], r['start_to_grip_s'], med_a])
    for _, r in b[b['H2b_flag']].iterrows():  # flagged Part B gaps
        details.append([int(e['event']), 'B release->oven', r['release'],
                        r['oven_start'], r['gap_s'], med_b])
print('\nH2 per event:')
h2_summary = pd.DataFrame(rows, columns=[  # H2 result per event
    'event', 'start_utc', 'end_utc', 'A_segments', 'A_flagged',
    'B_gaps', 'B_flagged'])
print(h2_summary.to_string(index=False))  # per-event table
if details:  # any flagged segment overlaps an event
    d = pd.DataFrame(details, columns=[
        'event', 'part', 'from_utc', 'to_utc', 'seconds', 'median_s'])
    print('\nFlagged H2 segments overlapping an event:')
    print(d.to_string(index=False, float_format=lambda x: f'{x:.3f}'))
    d.to_csv(FOLDER + 'b4_h2_per_event_flagged.csv', index=False)
print('\nAll flagged segments in the recording (Part A, Part B):')
print(A[A['H2a_flag']][['interval', 'belt_start', 'grip_on',
                        'start_to_grip_s']].to_string(
    index=False, float_format=lambda x: f'{x:.3f}'))
print(B[B['H2b_flag']].to_string(index=False,
                                 float_format=lambda x: f'{x:.3f}'))
print('\nTiming proxies, not verified physical cycles. '
      'A flag means longer than usual, not a confirmed loss.')


Part A: 8 complete belt-start intervals | median belt-start-to-grip 1.445 s | flagged: 1 | without grip: 0
Part B: 9 release-to-oven pairs | median gap 1.700 s | flagged: 3

H2 per event:
 event    start_utc      end_utc  A_segments  A_flagged  B_gaps  B_flagged
     1 18:21:16.045 18:21:20.040           0          0       0          0
     2 18:21:46.305 18:21:47.495           0          0       0          0
     3 18:22:57.450 18:22:58.895           0          0       0          0
     4 18:23:18.190 18:23:24.055           0          0       1          0
     5 18:24:16.245 18:24:18.030           0          0       0          0
     6 18:25:22.715 18:25:25.180           0          0       0          0
     7 18:25:31.980 18:25:38.695           0          0       0          0
     8 18:26:17.965 18:26:18.050           0          0       0          0
     9 18:26:58.170 18:26:59.530           0          0       0          0
    10 18:27:20.100 18:27:25.965           0          0       

**What:**
- H3: long pauses between completed parts. We pool the starts of the pusher outputs `A0.2` (white), `A0.3` (red) and `A0.4` (blue), sort them in time and measure the gap between consecutive starts
- Flag a gap that is above 1.25 times the median gap (the screening rule we use)
- For each event, count the gaps that overlap it, how many are flagged and the longest gap
- Needs `complete()` from cell 3

**Why:**
- The time between pusher activations shows if the plant keeps its usual rhythm. We use the pusher as a marker of a finished part, but we have not verified that one activation is one completed part
- The cell stops if fewer than 6 pusher activations exist

**Output:**
- 10 pooled pusher starts, 9 gaps, median 45.730 s, threshold 57.1625 s, 3 flagged: 71.655 s (18:22:12.910 to 18:23:24.565), 73.950 s (18:24:44.465 to 18:25:58.415) and 72.080 s (18:28:03.195 to 18:29:15.275)
- Events that overlap a flagged gap: 3, 4, 6, 7, 12 and 13 (one gap each). Events 2, 5, 8, 9, 10, 11 overlap one gap that is not flagged. Event 1 overlaps no gap, because it lies before the first pusher start
- Saved `b4_h3_gaps.csv`

**Next:** H4, the dictionary station of the flagged H1 signals.

In [19]:
# Cell 4: H3 per event (long pause between completed parts).
# Pusher outputs A0.2, A0.3, A0.4: complete activation starts pooled,
# gap between consecutive starts.
# Flag: gap above 1.25 x the median gap (inherited screening rule).
# Pusher = completion proxy, unverified.
starts = sorted(st for c in ('A0.2', 'A0.3', 'A0.4')  # pooled start times
                for st, _ in complete(c))
assert len(starts) >= 6, 'Too few complete pusher activations for a baseline.'
H3 = pd.DataFrame({'from_utc': starts[:-1], 'to_utc': starts[1:]})  # pairs
H3['gap_s'] = (H3['to_utc'] - H3['from_utc']).dt.total_seconds()  # gap
med_h3 = H3['gap_s'].median()  # usual gap
thr_h3 = 1.25 * med_h3  # flag threshold
H3['H3_flag'] = H3['gap_s'] > thr_h3  # long gap
print(f'{len(starts)} pooled pusher starts -> {len(H3)} gaps | '
      f'median {med_h3:.3f} s | threshold {thr_h3:.4f} s | '
      f'flagged: {int(H3["H3_flag"].sum())}')
print(H3.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

rows = []  # per-event summary
for _, e in events.iterrows():  # each event
    g = H3[(H3['from_utc'] < e['end_utc'])  # gaps overlapping the event
           & (H3['to_utc'] > e['start_utc'])]
    rows.append([int(e['event']),
                 e['start_utc'].strftime('%H:%M:%S.%f')[:-3],
                 e['end_utc'].strftime('%H:%M:%S.%f')[:-3],
                 len(g), int(g['H3_flag'].sum()),
                 round(g['gap_s'].max(), 3) if len(g) else np.nan])
h3_summary = pd.DataFrame(rows, columns=[  # H3 result per event
    'event', 'start_utc', 'end_utc', 'gaps_overlapping', 'H3_flagged',
    'longest_gap_s'])
print('\nH3 per event:')
print(h3_summary.to_string(index=False))  # per-event table
H3.to_csv(FOLDER + 'b4_h3_gaps.csv', index=False)  # save the gaps
print('\nPusher gaps are a completion proxy, not verified part '
      'completion. Flag = longer than usual, not a confirmed loss.')


10 pooled pusher starts -> 9 gaps | median 45.730 s | threshold 57.1625 s | flagged: 3
                        from_utc                           to_utc  gap_s  H3_flag
2019-09-26 18:21:44.435000+00:00 2019-09-26 18:22:12.910000+00:00 28.475    False
2019-09-26 18:22:12.910000+00:00 2019-09-26 18:23:24.565000+00:00 71.655     True
2019-09-26 18:23:24.565000+00:00 2019-09-26 18:23:50.745000+00:00 26.180    False
2019-09-26 18:23:50.745000+00:00 2019-09-26 18:24:44.465000+00:00 53.720    False
2019-09-26 18:24:44.465000+00:00 2019-09-26 18:25:58.415000+00:00 73.950     True
2019-09-26 18:25:58.415000+00:00 2019-09-26 18:26:44.145000+00:00 45.730    False
2019-09-26 18:26:44.145000+00:00 2019-09-26 18:27:26.815000+00:00 42.670    False
2019-09-26 18:27:26.815000+00:00 2019-09-26 18:28:03.195000+00:00 36.380    False
2019-09-26 18:28:03.195000+00:00 2019-09-26 18:29:15.275000+00:00 72.080     True

H3 per event:
 event    start_utc      end_utc  gaps_overlapping  H3_flagged  longest_gap_s


**What:**
- H4: read the Station and Comment columns of the signal dictionary (`Variablen*.xlsx`, sheets Beckhoff and Siemens) and add the station to every H1-flagged activation that overlaps an event ("Unmapped" if the dictionary has no station)
- Print these activations with their station, and the extra seconds and activation count per event and station
- Descriptive only: nothing is inferred beyond the dictionary entry
- Needs `h1` from cell 2

**Why:**
- To see which station the flagged signals belong to in each event. A station is not a proven bottleneck or loss
- The cell stops if there is not exactly one dictionary file or a sheet has no station column

**Output:**
- Event 7: Warehouse (`WH.Q3_horRack`, 10.497 s extra) and Drilling station (`A1.7`, 8.840 s extra)
- Event 10: Drilling station (`A0.7`, 2.210 s extra)
- Event 11: Warehouse (`WH.Q7_cantForw`, 10.370 s extra)
- Event 13: Warehouse (`WH.Q3_horRack`, 7.182 s summed extra, 7.183 s in the table above, a rounding difference)
- The dictionary comment is empty for the three Warehouse signals. Signals not in the dictionary: none
- Saved `b4_h4_station_per_event.csv`

**Next:** the evidence table per event.

In [20]:
# Cell 5: H4 per event (station of the flagged signals). Descriptive only.
# Needs h1 from cell 2. Station and comment text come from the signal
# dictionary, nothing is inferred.
import openpyxl  # reads the Excel dictionary
from pathlib import Path  # file paths

DICT_DIR = Path(  # folder with the signal dictionary
    '/Users/bala/Desktop/Albstadt Uni/Project B4/'
    'Bala\'s work/Example batch-Dataset')
files = sorted(DICT_DIR.glob('Variablen*.xlsx'))  # the dictionary file
assert len(files) == 1, f'Expected one dictionary file, found: {files}'
wb = openpyxl.load_workbook(files[0], read_only=True)  # open it


def lookup(sheet):  # signal -> (station, comment) for one sheet
    rows = list(wb[sheet].iter_rows(values_only=True))  # all rows
    head = [str(h).strip().lower() if h is not None else ''
            for h in rows[0]]  # header names in lower case
    assert 'station' in head, f'{sheet}: no station column in {head}'
    si = head.index('station')  # column of the station
    ci = head.index('comment') if 'comment' in head else None  # comment
    return {r[0]: ((r[si] or '').strip() or 'Unmapped',
                   (r[ci] or '').strip() if ci is not None else '')
            for r in rows[1:] if r[0]}  # skip rows without a signal name


dic = {**lookup('Beckhoff'), **lookup('Siemens')}  # both sheets together
wb.close()  # close the file

f = h1[h1['H1_flag']].copy()  # all H1-flagged activations
f['station'] = f['signal'].map(  # station from the dictionary
    lambda s: dic.get(s, ('Unmapped', ''))[0])
f['dictionary_comment'] = f['signal'].map(  # comment from the dictionary
    lambda s: dic.get(s, ('', ''))[1])
parts = []  # flagged activations per event
for _, e in events.iterrows():  # each event
    hit = f[(f['on_utc'] < e['end_utc'])  # flagged activations overlapping
            & (f['off_utc'] > e['start_utc'])]
    if len(hit):  # keep the event number
        parts.append(hit.assign(event=int(e['event'])))
if parts:  # at least one flagged activation overlaps an event
    t4 = pd.concat(parts)[[
        'event', 'station', 'signal', 'dictionary_comment',
        'on_utc', 'off_utc', 'activation_s', 'extra_s']]
    print('H1-flagged signals overlapping events, with dictionary station:')
    print(t4.sort_values(['event', 'on_utc']).to_string(
        index=False, float_format=lambda x: f'{x:.3f}'))
    print('\nPer event and station (summed extra seconds, count):')
    print(t4.groupby(['event', 'station']).agg(  # sums per event and station
        extra_s=('extra_s', 'sum'), activations=('signal', 'size'))
          .round(3).to_string())
    t4.to_csv(FOLDER + 'b4_h4_station_per_event.csv', index=False)  # save
else:
    print('No H1-flagged activation overlaps an event.')
miss = sorted(set(f['signal']) - set(dic))  # flagged signals not in dict
print('\nSignals not in the dictionary:', miss if miss else 'none')
print('Station = the dictionary entry for that signal. '
      'Descriptive, not a proven bottleneck or loss.')


H1-flagged signals overlapping events, with dictionary station:
 event          station         signal dictionary_comment                           on_utc                          off_utc  activation_s  extra_s
     7        Warehouse  WH.Q3_horRack                    2019-09-26 18:25:26.965000+00:00 2019-09-26 18:25:40.565000+00:00        13.600   10.497
     7 Drilling station           A1.7                    2019-09-26 18:25:28.665000+00:00 2019-09-26 18:25:46.770000+00:00        18.105    8.840
    10 Drilling station           A0.7                    2019-09-26 18:27:17.635000+00:00 2019-09-26 18:27:25.625000+00:00         7.990    2.210
    11        Warehouse WH.Q7_cantForw                    2019-09-26 18:27:42.030000+00:00 2019-09-26 18:27:55.460000+00:00        13.430   10.370
    13        Warehouse  WH.Q3_horRack                    2019-09-26 18:28:47.565000+00:00 2019-09-26 18:28:57.850000+00:00        10.285    7.183

Per event and station (summed extra seconds, count):


**What:**
- Build the evidence table per event: event window, flagged rows, `both_share` and the flags of H1 (cell 2), H2 (Part A or B, cell 3) and H3 (cell 4)
- `hypotheses_flagged` = how many of H1, H2, H3 are flagged for the event
- Reporting rule, set before the results: an event is "supported" if at least one of H1, H2 or H3 is flagged inside its window and the IF window is reproducible (`both_share` of at least 0.5, both independent 20-forest groups agree on its rows)
- Print the table, count the supported events and save it as `b4_event_evidence_table.csv`
- If more than 10 events had hypothesis support, the cell would print a note. We cut nothing

**Why:**
- One table for the report that shows for each of the 13 events which evidence exists, without picking a number of events
- Needs `summary` (cell 2), `h2_summary` (cell 3) and `h3_summary` (cell 4)

**Output:**
- 7 events are supported with a reproducible IF window: 3, 4, 6, 7, 10, 12 and 13
- Event 11 has a flag (H1) but its IF window is not reproducible (`both_share` 0.000): "H-supported, IF window not reproducible"
- 5 events are IF candidates only: 1, 2, 5, 8 and 9
- 8 of 13 events have at least one hypothesis flag. Events 7 and 13 have two flags (7: H1 and H3, 13: H1 and H3). H2 flags no event
- The "more than 10" note was not printed (8 events)

**Next:** the report and presentation, with all events labelled as candidates.

In [21]:
# Cell 6: evidence table per event and the reporting rule (set before the
# results). Supported = at least one of H1, H2 (A or B), H3 flagged
# overlapping the event window.
# IF window reproducible = both_share >= 0.5 (both independent 20-forest
# groups agree on its rows).
tab = (events[['event', 'start_utc', 'end_utc', 'flagged_rows',
               'both_share']]  # event window and IF information
       .merge(summary[['event', 'H1_flagged']], on='event')  # H1 flags
       .merge(h2_summary[['event', 'A_flagged', 'B_flagged']],
              on='event')  # H2 flags
       .merge(h3_summary[['event', 'H3_flagged']], on='event'))  # H3 flags
tab['hypotheses_flagged'] = (  # how many of H1, H2, H3 are flagged
    (tab['H1_flagged'] > 0).astype(int)
    + ((tab['A_flagged'] + tab['B_flagged']) > 0).astype(int)
    + (tab['H3_flagged'] > 0).astype(int))
tab['IF_reproducible'] = tab['both_share'] >= 0.5  # reproducible window
tab['status'] = np.where(  # reporting rule
    tab['hypotheses_flagged'] > 0,
    np.where(tab['IF_reproducible'], 'supported',
             'H-supported, IF window not reproducible'),
    'IF candidate only')
out = tab.assign(  # readable times for printing
    start_utc=tab['start_utc'].dt.strftime('%H:%M:%S.%f').str[:-3],
    end_utc=tab['end_utc'].dt.strftime('%H:%M:%S.%f').str[:-3])
print(out.to_string(index=False))  # the evidence table
n_sup = int((tab['status'] == 'supported').sum())  # supported events
n_h = int((tab['hypotheses_flagged'] > 0).sum())  # events with any flag
print(f'\nSupported with a reproducible IF window: {n_sup} | '
      f'with any hypothesis flag: {n_h} of {len(tab)} events')
if n_h > 10:  # we do not cut, but we say so
    print('More than 10 events have hypothesis support. '
          'Nothing is cut here; the reporting rule needs a decision.')
tab.to_csv(FOLDER + 'b4_event_evidence_table.csv', index=False)  # save
print('Saved: b4_event_evidence_table.csv. '
      'Candidates only, not confirmed losses.')


 event    start_utc      end_utc  flagged_rows  both_share  H1_flagged  A_flagged  B_flagged  H3_flagged  hypotheses_flagged  IF_reproducible                                  status
     1 18:21:16.045 18:21:20.040            47       1.000           0          0          0           0                   0             True                       IF candidate only
     2 18:21:46.305 18:21:47.495            15       1.000           0          0          0           0                   0             True                       IF candidate only
     3 18:22:57.450 18:22:58.895            13       1.000           0          0          0           1                   1             True                               supported
     4 18:23:18.190 18:23:24.055            35       0.875           0          0          0           1                   1             True                               supported
     5 18:24:16.245 18:24:18.030            21       0.923           0          0         